# LMA Phase 2 - evaluation from a pretrained checkpoint

Runs the full Phase 2 evaluation suite against an already-trained checkpoint,
without retraining. Separate from the pretraining notebook for two reasons:

1. The pretraining runs evaluated with an older revision of the code and without
   `sacrebleu` installed, so their BLEU / chrF++ / ROUGE-L metrics never got
   written. Training is the expensive part and is unaffected - the checkpoints
   are intact, and evaluation is cheap to redo.
2. Re-running both models through the *same* evaluation code is what makes the
   Model H / Model L comparison legitimate. A side-by-side table built from two
   different revisions would not be.

### Before running
- Accelerator: **GPU T4** (P100 is sm_60 and unsupported by current PyTorch).
- Attach: `lma-code`, this language's packed corpus, and the **notebook output**
  of that language's pretraining run (Add Input -> Notebook Output).

## 1. Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"], check=True)
import sacrebleu
print("sacrebleu:", sacrebleu.__version__)

## 2. Environment and code

In [ ]:
import os, json, shutil, zipfile, time
from pathlib import Path
import torch

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU  :", torch.cuda.get_device_name(0),
          "| cc %d.%d" % (props.major, props.minor))
    if (props.major, props.minor) < (7, 0):
        raise RuntimeError("compute capability below 7.0; request NvidiaTeslaT4")

INPUT_ROOT = Path("/kaggle/input")


def walk_dirs(root, max_depth=6):
    # Kaggle nests mounts and does not name them after the dataset slug, so
    # everything below is located by content rather than by path.
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        yield current
        if depth < max_depth:
            try:
                for child in sorted(current.iterdir()):
                    if child.is_dir():
                        stack.append((child, depth + 1))
            except OSError:
                pass


REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True)

src = None
for cand in walk_dirs(INPUT_ROOT):
    if (cand / "hindi").is_dir() and (cand / "nepali").is_dir():
        src = cand
        break
if src is None:
    raise FileNotFoundError("code dataset not found among attached inputs")
for item in sorted(src.iterdir()):
    target = REPO / item.name
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("code from", src)

## 3. Locate this language's checkpoint and corpus

In [ ]:
# ---- the only setting you change between the two runs ----
LANGUAGE = "hindi"          # "hindi" (Model H) or "nepali" (Model L)
CHECKPOINT_NAME = "ckpt_best.pt"
# ----------------------------------------------------------

# Corpus: the mount holding train.bin whose sidecar names THIS language. Testing
# the sidecar rather than the path is also what stops Model H ever being scored
# against Model L's data.
DATA_DIR = None
for cand in walk_dirs(INPUT_ROOT):
    sidecar = cand / "test.json"
    if (cand / "test.bin").is_file() and sidecar.is_file():
        if json.loads(sidecar.read_text())["language"] == LANGUAGE:
            DATA_DIR = cand
            break
if DATA_DIR is None:
    raise FileNotFoundError("no attached input holds the " + LANGUAGE + " corpus")

# Checkpoint: from the attached pretraining notebook output.
CKPT = None
for cand in walk_dirs(INPUT_ROOT):
    if cand.name == LANGUAGE and (cand / CHECKPOINT_NAME).is_file():
        CKPT = cand / CHECKPOINT_NAME
        break
if CKPT is None:
    for cand in walk_dirs(INPUT_ROOT):
        hit = cand / CHECKPOINT_NAME
        if hit.is_file():
            CKPT = hit
            break
if CKPT is None:
    raise FileNotFoundError("attach the pretraining notebook output for " + LANGUAGE)

# The tokenizer must sit where the evaluator looks for it.
tok_dir = REPO / LANGUAGE / "tokenizer"
tok_dir.mkdir(parents=True, exist_ok=True)
for pattern in ("*.model", "*.vocab"):
    for f in sorted(DATA_DIR.glob(pattern)):
        shutil.copy2(f, tok_dir / f.name)

print("language  :", LANGUAGE)
print("corpus    :", DATA_DIR)
print("checkpoint:", CKPT, "(%.0f MB)" % (CKPT.stat().st_size / 1e6))

## 4. Full evaluation suite

Intrinsic metrics on **both** validation and test (the spec asks for validation
cross-entropy and perplexity; test is the cleaner final number, so both are
reported), plus BLEU / chrF++ / ROUGE-L at greedy and temperatures 0.5, 1.0,
1.5, repetition and Distinct-n, and the attention statistics.

In [ ]:
import importlib
ev_mod = importlib.import_module(LANGUAGE + ".eval.evaluate")

started = time.time()
ev = ev_mod.Evaluator(str(CKPT), data_dir=DATA_DIR)
report = ev.full_report(split="test")
out = Path("/kaggle/working/" + LANGUAGE + "_eval.json")
out.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print("evaluated in %.1f min -> %s" % ((time.time() - started) / 60, out))

print()
print("checkpoint step:", format(report["checkpoint_step"], ","))
for split, m in report["intrinsic_by_split"].items():
    print("  %-4s  CE %.4f  PPL %8.2f  BPB %.4f  (%s tokens)"
          % (split, m["cross_entropy_nats"], m["perplexity"],
             m["bits_per_byte"], format(m["tokens_scored"], ",")))
print()
for name, m in report["generation"]["settings"].items():
    print("  %-16s BLEU %6.2f  chrF++ %6.2f  ROUGE-L %.3f  rep %.3f  d1 %.3f  d2 %.3f"
          % (name, m["bleu"], m["chrf_pp"], m["rouge_l"],
             m["repetition_rate"], m["distinct_1"], m["distinct_2"]))

## 5. Generated samples

In [ ]:
for s in report["generation"]["samples"][:6]:
    print("[" + s["setting"] + "]")
    print("  prompt   :", s["prompt"][:160].replace(chr(10), " "))
    print("  generated:", s["generated"][:160].replace(chr(10), " "))
    print("  reference:", s["reference"][:160].replace(chr(10), " "))
    print()

## 6. Attention figures

At least one early layer and one late layer, several heads each, with the
example sentence recorded as text so the figure is interpretable even where no
Devanagari font is installed.

In [ ]:
subprocess.run("apt-get -qq install -y fonts-noto-devanagari > /dev/null 2>&1",
               shell=True)
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)

attn_mod = importlib.import_module(LANGUAGE + ".eval.attention")
viz_mod = importlib.import_module(LANGUAGE + ".eval.viz")

FIGDIR = Path("/kaggle/working/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

x, _ = ev.data.fixed_batches("test", 4, 1, seed=11)[0]
attentions = attn_mod.collect_attention(ev.model, x)
tokens = [ev.decode([int(t)]) for t in x[0].cpu().numpy()]

for layer in (0, len(attentions) - 1):
    viz_mod.plot_attention_heatmap(
        attentions[layer][0].numpy(), tokens, layer, [0, 1, 2, 3],
        FIGDIR / ("%s_attention_layer%d.png" % (LANGUAGE, layer)),
        language=LANGUAGE)

summary = report["attention"]
viz_mod.plot_attention_summary({LANGUAGE: summary},
                               FIGDIR / (LANGUAGE + "_attention_summary.png"))
Path("/kaggle/working/" + LANGUAGE + "_attention.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False))

print("mean entropy    :", round(summary["mean_entropy"], 4))
print("uniform baseline:", round(summary["uniform_baseline_entropy"], 4))
print("mean distance   :", round(summary["mean_distance"], 2))
print()
print("example sentence visualised:")
print(" ", summary["example_sentences"][0][:300])

from IPython.display import Image, display
display(Image(str(FIGDIR / (LANGUAGE + "_attention_layer0.png"))))
display(Image(str(FIGDIR / (LANGUAGE + "_attention_summary.png"))))

## 7. Bundle for the repository

In [ ]:
bundle = Path("/kaggle/working/report_phase2.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in FIGDIR.glob("*.png"):
        zf.write(p, "figures/" + p.name)
    for name in (LANGUAGE + "_eval.json", LANGUAGE + "_attention.json"):
        p = Path("/kaggle/working") / name
        if p.exists():
            zf.write(p, name)
print("report_phase2.zip  %.2f MB -> commit into report/phase2/"
      % (bundle.stat().st_size / 1e6))